# It's Not About The Tool: Python (pandas) — four ONS tables, five questions, one dashboard
Data: ONS, *Alcohol-specific deaths in the UK: registered in 2024*, sheets **Table_6** (England regions), **Table_9** (causes by age, UK), **Table_10** (cause shares, UK), **Table_11** (deprivation, England).

Run every cell top to bottom.

## Step 1: Upload the workbook

In [ ]:
from google.colab import files
up = files.upload()
FILE = list(up.keys())[0]

## Step 2: Clean all four tables
**Why these functions:** `find_header` searches for the first real header row, so the code survives ONS adding a note line. `pd.to_numeric(errors="coerce")` turns anything that is not a number into blank instead of crashing. `.str.strip()` removes trailing spaces that would split "London " from "London". `dropna(subset=["Year"])` drops footnote rows.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

AGE = ['<1','01-04','05-09','10-14','15-19','20-24','25-29','30-34','35-39','40-44',
       '45-49','50-54','55-59','60-64','65-69','70-74','75-79','80-84','85-89','90+']

def find_header(sheet, first_word):
    raw = pd.read_excel(FILE, sheet_name=sheet, header=None)
    return int(raw.index[raw[0].astype(str).str.startswith(first_word)][0])

def tidy(df, nums, texts):
    df = df.dropna(how='all').copy()
    for c in nums:  df[c] = pd.to_numeric(df[c], errors='coerce')
    for c in texts: df[c] = df[c].astype(str).str.strip()
    return df.dropna(subset=['Year']).assign(Year=lambda d: d['Year'].astype(int))

t6 = pd.read_excel(FILE, sheet_name='Table_6', header=find_header('Table_6','Year of death'))
t6.columns = ['Year','Sex','AreaCode','Region','Deaths','Rate','LCL','UCL']
t6 = tidy(t6, ['Year','Deaths','Rate','LCL','UCL'], ['Sex','AreaCode','Region'])
t6['Deaths'] = t6['Deaths'].astype(int)

t9 = pd.read_excel(FILE, sheet_name='Table_9', header=find_header('Table_9','Area code'))
t9.columns = ['AreaCode','Area','Year','Sex','ICD','Cause'] + AGE + ['Total']
t9 = tidy(t9, ['Year','Total']+AGE, ['AreaCode','Area','Sex','ICD','Cause'])

t10 = pd.read_excel(FILE, sheet_name='Table_10', header=find_header('Table_10','Area code'))
t10.columns = ['AreaCode','Area','Year','Sex','ICD','Cause','Proportion']
t10 = tidy(t10, ['Year','Proportion'], ['AreaCode','Area','Sex','ICD','Cause'])

t11 = pd.read_excel(FILE, sheet_name='Table_11', header=find_header('Table_11','Year of death'))
t11.columns = ['Year','IMD','M_Deaths','M_Rate','M_LCL','M_UCL','F_Deaths','F_Rate','F_LCL','F_UCL']
t11 = tidy(t11, t11.columns.tolist(), [])

print("Rows:", len(t6), len(t9), len(t10), len(t11), " (expect 648, 1080, 1080, 35)")

## Q1. Where is it worst? (Table_6, England regions, 2024)
`sort_values` ranks; the LCL and UCL columns tell you whether a gap is real.

In [ ]:
p6  = t6[t6.Sex == 'Persons']
q1  = p6[p6.Year == 2024].sort_values('Rate', ascending=False)
print("Gap North East vs London is real:", q1.iloc[0].LCL > q1.iloc[-1].UCL,
      "| ratio", round(q1.iloc[0].Rate/q1.iloc[-1].Rate, 2))
q1[['Region','Deaths','Rate','LCL','UCL']]

## Q2. Which way is it going? (Table_6, 2001 to 2024)
`groupby` + `sum` is a PivotTable in one line. Counts can be added; rates cannot.

In [ ]:
q2 = p6.groupby('Year', as_index=False)['Deaths'].sum()
g  = q2.set_index('Year')['Deaths']
print("2019:", g[2019], "| 2023 peak:", g[2023], "| 2024:", g[2024],
      "| vs 2019:", round((g[2024]/g[2019]-1)*100,1), "% | vs 2023:", round((g[2024]/g[2023]-1)*100,1), "%")
q2.tail(7)

## Q3. What are people dying of? (Table_9 counts and Table_10 shares, UK)
`merge` joins the count table to the share table on the same keys, so one row carries both.

In [ ]:
c24 = t9[(t9.Year==2024) & (t9.Sex=='Persons')][['ICD','Cause','Total']]
s24 = t10[(t10.Year==2024) & (t10.Sex=='Persons')][['ICD','Proportion']]
q3  = c24.merge(s24, on='ICD').sort_values('Total', ascending=False)
print("UK deaths 2024:", int(q3.Total.sum()), "(England regions, Table_6: ", int(g[2024]), ")")
liver = t10[(t10.ICD=='K70') & (t10.Sex=='Persons')][['Year','Proportion']].sort_values('Year')
print("Alcoholic liver disease share: 2001 =", float(liver[liver.Year==2001].Proportion.iloc[0]),
      "% -> 2024 =", float(liver[liver.Year==2024].Proportion.iloc[0]), "%")
q3.head(5)

## Q4. Which ages? (Table_9 age bands, UK, 2024)
The 20 age columns are summed, then turned into percentages so the shape is comparable between sexes.

In [ ]:
ages = t9[(t9.Year==2024) & (t9.Sex=='Persons')][AGE].sum()
q4 = (ages/ages.sum()*100).round(1)
print("Peak band:", q4.idxmax(), q4.max(), "% | aged 45-69:", round(q4[['45-49','50-54','55-59','60-64','65-69']].sum(),1),
      "% | under 45:", round(q4[AGE[:10]].sum(),1), "%")
q4[q4 > 0]

## Q5. Does deprivation explain it? (Table_11, England, 2024)
One row per quintile, so the ratio of the most to the least deprived is a single division.

In [ ]:
q5 = t11[t11.Year==2024].sort_values('IMD')
print("Male rate, most vs least deprived:", q5.iloc[0].M_Rate, "vs", q5.iloc[-1].M_Rate,
      "=", round(q5.iloc[0].M_Rate/q5.iloc[-1].M_Rate,2), "times")
print("Female rate, most vs least deprived:", q5.iloc[0].F_Rate, "vs", q5.iloc[-1].F_Rate,
      "=", round(q5.iloc[0].F_Rate/q5.iloc[-1].F_Rate,2), "times")
print("Cross-check: deaths in Table_11 2024 =", int(q5.M_Deaths.sum()+q5.F_Deaths.sum()), "(Table_6 says", int(g[2024]), ")")
q5[['IMD','M_Deaths','M_Rate','F_Deaths','F_Rate']]

## The dashboard (one image, portfolio ready)

In [ ]:
NAVY, RED, ORANGE, GREY, INK, MUTED = '#2C3E50', '#C0392B', '#E59866', '#95A5A6', '#1F2A36', '#6B7785'
plt.rcParams.update({'font.family':'DejaVu Sans','axes.edgecolor':'#D7DCE1','text.color':INK,
                     'axes.labelcolor':MUTED,'xtick.color':MUTED,'ytick.color':MUTED,'axes.titlecolor':NAVY})

fig = plt.figure(figsize=(16, 9), facecolor='white')
gs  = GridSpec(3, 3, figure=fig, height_ratios=[0.52, 1, 1], hspace=0.75, wspace=0.30,
               left=0.085, right=0.97, top=0.88, bottom=0.11)

fig.text(0.06, 0.955, 'Alcohol-specific deaths: place, trend, cause, age and deprivation',
         fontsize=22, fontweight='bold', color=NAVY)
fig.text(0.06, 0.925, 'ONS, Alcohol-specific deaths in the UK: registered in 2024 (Tables 6, 9, 10 and 11). '
                      'Rates are age-standardised per 100,000.', fontsize=10, color=MUTED)

# KPI strip
kpis = [(f"{int(g[2024]):,}", "deaths, England regions, 2024"),
        (f"+{round((g[2024]/g[2019]-1)*100,1)}%", "vs 2019, before the pandemic"),
        (f"{q3.iloc[0].Proportion:.0f}%", "of UK deaths: alcoholic liver disease"),
        (f"{round(q5.iloc[0].M_Rate/q5.iloc[-1].M_Rate,1)}x", "male rate, most vs least deprived")]
ax = fig.add_subplot(gs[0, :]); ax.axis('off')
for i, (num, lab) in enumerate(kpis):
    x = i * 0.25
    ax.add_patch(plt.Rectangle((x, 0.05), 0.235, 0.9, transform=ax.transAxes,
                               facecolor='#F4F6F8', edgecolor='none'))
    ax.text(x+0.02, 0.58, num, transform=ax.transAxes, fontsize=26, fontweight='bold',
            color=RED if i < 2 else NAVY, va='center')
    ax.text(x+0.02, 0.22, lab, transform=ax.transAxes, fontsize=10, color=INK, va='center')

# Panel 1: regions with 95% ranges
ax1 = fig.add_subplot(gs[1, 0]); d = q1.sort_values('Rate')
ax1.barh([r.replace('Yorkshire and The Humber','Yorkshire & Humber') for r in d.Region], d.Rate, color=[RED if r == 'North East' else GREY for r in d.Region])
ax1.errorbar(d.Rate, [r.replace('Yorkshire and The Humber','Yorkshire & Humber') for r in d.Region], xerr=[d.Rate-d.LCL, d.UCL-d.Rate], fmt='none', ecolor=INK, capsize=2, lw=1)
ax1.set_title('Where: rate by region, 2024', fontsize=12, loc='left', pad=8)
ax1.set_xlabel('per 100,000 (lines = 95% range)', fontsize=9)
ax1.tick_params(labelsize=8); [s.set_visible(False) for s in (ax1.spines['top'], ax1.spines['right'])]

# Panel 2: trend
ax2 = fig.add_subplot(gs[1, 1])
ax2.plot(q2.Year, q2.Deaths, color=RED, lw=2, marker='o', ms=3)
ax2.axvspan(2019.5, 2021.5, color=GREY, alpha=0.18)
ax2.annotate(f"{int(g[2023]):,}", (2023, g[2023]), textcoords='offset points', xytext=(0,8), ha='center', fontsize=8)
ax2.set_title('When: deaths per year, England regions', fontsize=12, loc='left', pad=8)
ax2.tick_params(labelsize=8); [s.set_visible(False) for s in (ax2.spines['top'], ax2.spines['right'])]

# Panel 3: causes
ax3 = fig.add_subplot(gs[1, 2]); c = q3.head(5).sort_values('Proportion')
ypos = np.arange(len(c))
ax3.barh(ypos, c.Proportion, color=[GREY]*4+[RED], height=0.5)
for y, v, name in zip(ypos, c.Proportion, c.Cause):
    short = name if len(name) < 40 else name[:37]+'...'
    ax3.text(0, y+0.36, short, fontsize=8, color=INK, va='bottom')
    ax3.text(v+1.5, y, f"{v:.1f}%", va='center', fontsize=8, fontweight='bold')
ax3.set_xlim(0, 95); ax3.set_ylim(-0.6, len(c)-0.2); ax3.set_yticks([])
ax3.set_title('What: share of UK deaths by cause, 2024', fontsize=12, loc='left', pad=8)
ax3.tick_params(labelsize=8); [s.set_visible(False) for s in (ax3.spines['top'], ax3.spines['right'], ax3.spines['left'])]

# Panel 4: ages
ax4 = fig.add_subplot(gs[2, 0]); a = q4[q4 > 0]
ax4.bar(a.index, a.values, color=[RED if i == a.idxmax() else NAVY for i in a.index])
ax4.set_title('Who: age at death, UK, 2024 (% of deaths)', fontsize=12, loc='left', pad=8)
ax4.tick_params(axis='x', rotation=60, labelsize=7); ax4.tick_params(axis='y', labelsize=8)
[s.set_visible(False) for s in (ax4.spines['top'], ax4.spines['right'])]

# Panel 5: deprivation
ax5 = fig.add_subplot(gs[2, 1]); x = np.arange(5)
ax5.bar(x-0.2, q5.M_Rate, 0.4, label='Men', color=NAVY)
ax5.bar(x+0.2, q5.F_Rate, 0.4, label='Women', color=ORANGE)
ax5.set_xticks(x); ax5.set_xticklabels(['1\nmost\ndeprived','2','3','4','5\nleast\ndeprived'], fontsize=8)
ax5.legend(frameon=False, fontsize=8)
ax5.set_title('Why it is unequal: rate by deprivation, England 2024', fontsize=12, loc='left', pad=8)
ax5.tick_params(labelsize=8); [s.set_visible(False) for s in (ax5.spines['top'], ax5.spines['right'])]

# Panel 6: liver share over time
ax6 = fig.add_subplot(gs[2, 2])
ax6.plot(liver.Year, liver.Proportion, color=NAVY, lw=2)
ax6.set_ylim(60, 85)
ax6.set_title('Cause mix: liver disease share of UK deaths', fontsize=12, loc='left', pad=8)
ax6.tick_params(labelsize=8); [s.set_visible(False) for s in (ax6.spines['top'], ax6.spines['right'])]

fig.text(0.06, 0.015, 'Table_6 and Table_11 are England; Table_9 and Table_10 are the whole UK. '
                     'Year of registration, not year of death. Built by Paula.', fontsize=8.5, color=MUTED)
plt.savefig('dashboard_python.png', dpi=170, facecolor='white')
plt.show()

## Download the dashboard

In [ ]:
files.download('dashboard_python.png')